#### Week 11 workshop solution for Tasks 2 and 3.
<author>&copy; Professor Yuefeng Li </author>

SVM classifiers and NB classifier.

In [12]:
# Analyze how SVC constructs non-linear decision boundaries via support vectors
# Examine how LinearSVC scores and ranks samples using the decision_function

In [14]:
# Task 2 (b) solution

import sys
import string

if __name__ == '__main__':
    from sklearn import svm
    from sklearn.metrics import confusion_matrix
    
    X_train=[[0, 0, 0, 0, 2], [3, 0, 1, 0, 1], [0, 0, 0, 0, 1], [2, 0, 3, 0, 2], [5, 2, 0, 0, 1], [0, 0, 1, 0, 1], [0, 1, 1, 0, 1], [0, 0, 0, 0, 1], [0, 0, 0, 0, 1], [1, 1, 0, 1, 2]]
    y_train=[-1, 1, -1, 1, 1, -1, -1, -1, -1, -1]


    X_test=[[1, 1, 1, 1, 2], [3, 0, 2, 0, 3], [0, 0, 0, 0, 0], [6, 0, 5, 0, 1], [4, 2, 0, 2, 1], [0, 0, 1, 1, 1], [0, 1, 0, 0, 1], [1, 0, 0, 0, 1], [0, 1, 0, 0, 1], [1, 1, 0, 1, 2]]
    y_test=[-1, 1, 1, 1, 1, -1, -1, -1, -1, -1]

    clf = svm.SVC(gamma='scale') # Supports kernel trick (non-linear decision boundaries)
    clf.fit(X_train, y_train)
    result1 = clf.predict(X_test)
    print("SVC Predication Results: " + str(result1))
    supp_v = clf.support_vectors_
    print("Support Vectors for SVC: " + str(supp_v))
  
    lin_clf = svm.LinearSVC() # Much faster for large datasets than SVC
    fit=lin_clf.fit(X_train, y_train)
    result2 = fit.predict(X_test)
    print("Linear Predication Results: s" + str(result2))
    
    dec = lin_clf.decision_function(X_test)
    print("Linear decision function values: " + str(dec))

    print("The Confusion Matrix for Result1::")
    print(confusion_matrix(y_test, result1))
    
    print("The Confusion Matrix for Result2::")
    print(confusion_matrix(y_test, result2))

SVC Predication Results: [-1  1 -1  1  1 -1 -1 -1 -1 -1]
Support Vectors for SVC: [[0. 0. 0. 0. 2.]
 [0. 0. 1. 0. 1.]
 [0. 1. 1. 0. 1.]
 [1. 1. 0. 1. 2.]
 [3. 0. 1. 0. 1.]
 [2. 0. 3. 0. 2.]
 [5. 2. 0. 0. 1.]]
Linear Predication Results: s[-1  1 -1  1  1 -1 -1 -1 -1 -1]
Linear decision function values: [-0.68288285  0.83899597 -0.63294057  4.31150494  1.01572447 -0.75644495
 -1.2034757  -0.33203565 -1.2034757  -0.96749374]
The Confusion Matrix for Result1::
[[6 0]
 [1 3]]
The Confusion Matrix for Result2::
[[6 0]
 [1 3]]


In [16]:
# Task 3 solution
# The following two functions come from the solution to Review Question 2 in Week 9. 

In [18]:
# Define function to calculate the number of words in documents that labelled as class c in D
def n_words(c,C,D):
    n_w = 0
    for i in range(len(C)):
        if C[i]==c:
            for j in range(len(D[i])):
                n_w = n_w + D[i][j] 
    return n_w

# The training algorithm, where we assume there are only two classes. 
# It is different to the one defined in the lecture notes. It is good to understand the differences.
# For example, here we assume the document parsing is done and the set of terms V is provided.  

def TRAIN_MULTINOMIAL_NB(C, D, V):
    # initialize variables
    N = len(D)
    prior = []
    condprob = tf = [[0 for w in range(len(V))] for c in range(2)]
    for c in range(2):
        Nc = 0
        for i in C:
            if C[i]==c:
                Nc = Nc+1
        prior.append(Nc/len(C))
        for w in range(len(V)):  # calculate tf(c,w)
            for d in range(len(D)):
                if C[d]==c:
                    tf[c][w] = tf[c][w] + D[d][w]
        for w in range(len(V)):
            condprob[c][w] = (tf[c][w]+1)/(n_words(c,C,D)+len(V))
    return(prior, condprob)
    

In [20]:
# Task 3 (b), design a function that uses the (prior, condprob) returned from training to assign a label to document d, 

import math
def Test_MNB(prior, condprob, d):
    score = {}
    for c in range(2):
        score[c]=math.log(prior[c])
        for i in range(len(d)):
            if d[i]>0:
                score[c] = score[c] + math.log(condprob[c][i])
    if score[1]>score[0]:
        y = 1
    else:
        y = -1
    return y

In [22]:
# The training dataset
X_train=[[0, 0, 0, 0, 2], [3, 0, 1, 0, 1], [0, 0, 0, 0, 1], [2, 0, 3, 0, 2], [5, 2, 0, 0, 1], [0, 0, 1, 0, 1], [0, 1, 1, 0, 1], [0, 0, 0, 0, 1], [0, 0, 0, 0, 1], [1, 1, 0, 1, 2]]
y_train=[-1, 1, -1, 1, 1, -1, -1, -1, -1, -1]
V = ['cheep', 'buy', 'banking', 'dinner', 'the']  # only for calling function TRAIN_MULTINOMIAL_NB()
# transfer y_train to my_train in the format of [0, 1, 0, 1, 1, 0, 0, 0, 0, 0]
my_train = y_train
for i in range(len(y_train)):
    if y_train[i]<1:
        my_train[i]=0

# The test dataset
X_test=[[1, 1, 1, 1, 2], [3, 0, 2, 0, 3], [0, 0, 0, 0, 0], [6, 0, 5, 0, 1], [4, 2, 0, 2, 1], [0, 0, 1, 1, 1], [0, 1, 0, 0, 1], [1, 0, 0, 0, 1], [0, 1, 0, 0, 1], [1, 1, 0, 1, 2]]
# the class lables for the test dataset 
y_test=[-1, 1, 1, 1, 1, -1, -1, -1, -1, -1]

# Task 3 (a) - Train the MULTINOMIAL NB classifier
(prior, condprob) = TRAIN_MULTINOMIAL_NB(my_train, X_train, V)
print('Probabilities p(c):', prior)
print('Conditional probabilities p(wi|c):', condprob)

# Task 3 (c) - Test the classifier for all documents 
test_result = []
for d in X_test:
    y = Test_MNB(prior, condprob, d)
    test_result = test_result + [y]
print('Test result:', test_result)

Probabilities p(c): [0.7, 0.3]
Conditional probabilities p(wi|c): [[0.1, 0.15, 0.15, 0.1, 0.5], [0.44, 0.12, 0.2, 0.04, 0.2]]
Test result: [-1, 1, -1, 1, -1, -1, -1, -1, -1, -1]


In [24]:
# Task 3 (d) - Design list comprehensions to compute TP, TN, FP, and FN;
TP = len([i for i in range(len(y_test)) if y_test[i]==1 and test_result[i]==1])
TN = len([i for i in range(len(y_test)) if y_test[i]==-1 and test_result[i]==-1])
FP = len([i for i in range(len(y_test)) if y_test[i]==1 and test_result[i]==-1])
FN = len([i for i in range(len(y_test)) if y_test[i]==-1 and test_result[i]==1])
# You may get TP = 2, TN = 6, FP = 2, FN =0

# Accuracy 
Acc_MNB = (TP+TN)/(TP+TN+FP+FN)
print('Accuracy:', Acc_MNB)
      

Accuracy: 0.8
